# 📚 Technique 58: Re-Ranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/58_re_ranking.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 58
**Difficulty:** Advanced

## 📋 Description

**Re-Ranking** is a two-stage retrieval technique where an initial retrieval method (fast but less accurate) fetches a candidate set of documents, and a more powerful re-ranking model (slower but more accurate) reorders these candidates to improve result quality. This approach balances efficiency and effectiveness.

### When to Use:
- When you need **high-quality ranking** but have **large document collections**
- For **improving retrieval accuracy** without reindexing
- When initial retriever is **fast but coarse** (e.g., vector similarity)
- For **cross-encoder scoring** that captures query-document interactions
- When you want to **add new ranking signals** without changing the index

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                   RE-RANKING PIPELINE                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  STAGE 1: INITIAL RETRIEVAL (Fast, Approximate)                 │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │   User      │───▶│   Dense      │───▶│  Candidate Set  │    │
│  │   Query     │    │   Retrieval  │    │  (Top 100)      │    │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│         │                  │                     │              │
│         │                  ▼                     ▼              │
│         │         ┌─────────────────┐    ┌──────────────┐       │
│         │         │  Vector Search  │    │  Fast but    │       │
│         │         │  (Cosine Sim)   │    │  coarse      │       │
│         │         └─────────────────┘    └──────────────┘       │
│                                                                 │
│                              │                                  │
│                              ▼                                  │
│  STAGE 2: RE-RANKING (Slow, Accurate)                           │
│  ┌─────────────────┐    ┌──────────────┐    ┌─────────────────┐│
│  │  Candidate Set  │───▶│  Cross-      │───▶│  Final Ranked   ││
│  │  (Top 100)      │    │  Encoder     │    │  Results (Top 5)││
│  └─────────────────┘    └──────────────┘    └─────────────────┘│
│         │                      │                     │          │
│         │                      ▼                     ▼          │
│         │             ┌─────────────────┐    ┌──────────────┐   │
│         │             │  Query + Doc    │    │  Precise     │   │
│         │             │  Joint Encoding │    │  Relevance   │   │
│         │             └─────────────────┘    └──────────────┘   │
│                                                                 │
│  KEY INSIGHT:                                                   │
│  • Stage 1: O(N) or O(log N) - scans millions of docs           │
│  • Stage 2: O(k) - only processes k candidates (e.g., 100)      │
│  • Total: Fast overall with high accuracy                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Re-Ranking Models:
| Model | Type | Speed | Accuracy | Use Case |
|-------|------|-------|----------|----------|
| **Cross-Encoder** | Full attention | Slow | Highest | Production RAG |
| **ColBERT** | Late interaction | Medium | High | Large-scale |
| **MonoT5** | Seq2seq | Medium | High | Research |
| **LLM-based** | GPT/Claude | Very Slow | Very High | Precision tasks |

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai numpy scikit-learn

In [ ]:
import os
from getpass import getpass
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

# Setup API
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector for text"""
    response = client.embeddings.create(model=model, input=text)
    return np.array(response.data[0].embedding)

def llm_score(query, document, model="gpt-3.5-turbo"):
    """Use LLM to score relevance (0-10)"""
    prompt = f"""Rate the relevance of the document to the query on a scale of 0-10.

Query: {query}

Document: {document}

Respond with ONLY a number from 0 to 10, where:
0 = Completely irrelevant
5 = Somewhat relevant
10 = Perfectly relevant

Score:"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 5.0  # default fallback

## 💡 Basic Example

Simple re-ranking with LLM-based scoring.

In [ ]:
# Sample document collection
documents = [
    "Python is a high-level programming language created by Guido van Rossum in 1991",
    "Python snakes are non-venomous constrictors found in Africa and Asia",
    "Monty Python was a British comedy group that created Flying Circus",
    "Python 3.10 introduced pattern matching and better error messages",
    "The reticulated python is the world's longest snake species",
    "Python is widely used for data science, AI, and web development",
    "Python eggs are incubated by the female for about 2-3 months",
    "Django and Flask are popular Python web frameworks"
]

print("Building initial retrieval index...")
doc_embeddings = np.array([get_embedding(doc) for doc in documents])
print(f"✓ Indexed {len(documents)} documents\n")

def initial_retrieval(query, doc_embeddings, documents, top_k=5):
    """Stage 1: Fast vector similarity retrieval"""
    query_embedding = get_embedding(query)
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    return [(idx, float(similarities[idx])) for idx in top_indices]

def llm_rerank(query, candidates, documents):
    """Stage 2: LLM-based re-ranking"""
    scored = []
    for idx, initial_score in candidates:
        llm_score_val = llm_score(query, documents[idx])
        # Combine initial and LLM scores
        combined = 0.3 * initial_score + 0.7 * (llm_score_val / 10)
        scored.append((idx, initial_score, llm_score_val, combined))
    
    # Sort by combined score
    scored.sort(key=lambda x: x[3], reverse=True)
    return scored

# Test query
query = "programming language features"

print(f"Query: '{query}'\n")

# Stage 1: Initial retrieval
print("=== STAGE 1: INITIAL RETRIEVAL (Vector Similarity) ===")
candidates = initial_retrieval(query, doc_embeddings, documents, top_k=5)
for idx, score in candidates:
    print(f"Score: {score:.4f} | {documents[idx][:60]}...")

# Stage 2: Re-ranking
print("\n=== STAGE 2: LLM RE-RANKING ===")
print("(This may take a moment as we call the LLM for each candidate...)\n")

reranked = llm_rerank(query, candidates, documents)

print(f"{'Rank':<6}{'Initial':<10}{'LLM':<8}{'Combined':<10} Document{'':<30}")
print("-" * 70)
for rank, (idx, init, llm, combined) in enumerate(reranked, 1):
    doc_short = documents[idx][:45] + "..." if len(documents[idx]) > 45 else documents[idx]
    print(f"{rank:<6}{init:<10.3f}{llm:<8.1f}{combined:<10.3f} {doc_short}")

print("\n✓ Notice how documents about the programming language moved up!")

## 🌍 Real-World Example

Legal document search with multi-stage re-ranking.

In [ ]:
# Legal document collection
legal_docs = [
    {
        "id": "CASE001",
        "title": "Smith v. TechCorp - Data Breach Class Action",
        "content": "Plaintiffs allege TechCorp failed to implement adequate security measures, resulting in unauthorized access to 2 million customer records. Court found negligence in security protocols. Settlement: $50M."
    },
    {
        "id": "CASE002",
        "title": "Johnson v. HealthPlus - Medical Records Privacy",
        "content": "Healthcare provider disclosed patient medical records to third parties without consent. Violated HIPAA regulations. Damages awarded: $2.3M for emotional distress and privacy violations."
    },
    {
        "id": "CASE003",
        "title": "Privacy Alliance v. SocialNet - GDPR Compliance",
        "content": "EU regulators fined SocialNet $270M for transferring user data to US servers without adequate safeguards. Landmark GDPR enforcement case establishing data localization requirements."
    },
    {
        "id": "CASE004",
        "title": "Doe v. FinanceBank - Unauthorized Account Access",
        "content": "Bank's mobile app vulnerability allowed attackers to access customer accounts. Court ruled bank liable for failing to patch known security flaw. Class action status granted."
    },
    {
        "id": "CASE005",
        "title": "Employee Union v. RetailCorp - Workplace Surveillance",
        "content": "Company installed hidden cameras in employee break rooms without notice. Violated state privacy laws and employee rights. Injunction granted, $5M in penalties."
    },
    {
        "id": "CASE006",
        "title": "State v. CyberCriminal - Identity Theft Prosecution",
        "content": "Defendant convicted of stealing identities through phishing schemes. 50 counts of identity theft, 10 years imprisonment. Restitution: $1.2M to victims."
    },
    {
        "id": "CASE007",
        "title": "Consumer Rights v. DataBroker - Sale of Personal Information",
        "content": "Data broker sold consumer personal information including SSNs and financial data to marketers without consent. CCPA violation. $30M fine imposed."
    },
    {
        "id": "CASE008",
        "title": "In re: CloudProvider - Government Data Request",
        "content": "Cloud provider challenged government subpoena for user data as overly broad. Court balanced privacy rights against law enforcement needs. Narrowed scope of disclosure."
    }
]

# Build index
legal_texts = [f"{d['title']}: {d['content']}" for d in legal_docs]
legal_embeddings = np.array([get_embedding(text) for text in legal_texts])

def legal_search_with_reranking(query, initial_k=6, final_k=3):
    """Legal search with re-ranking"""
    
    # Stage 1: Initial retrieval
    query_embedding = get_embedding(query)
    similarities = cosine_similarity([query_embedding], legal_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:initial_k]
    
    candidates = [(idx, float(similarities[idx])) for idx in top_indices]
    
    # Stage 2: LLM re-ranking with legal expertise
    rerank_prompt_template = """You are a legal research assistant. Rate how relevant this case is to the research query.

Research Query: {query}

Case: {case_title}
Case Summary: {case_content}

Rate relevance 0-10 where:
0 = Completely irrelevant to the legal issue
5 = Related but different legal context
10 = Directly on point, same legal issue

Respond with ONLY the numeric score:"""
    
    scored = []
    for idx, init_score in candidates:
        case = legal_docs[idx]
        prompt = rerank_prompt_template.format(
            query=query,
            case_title=case['title'],
            case_content=case['content']
        )
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        
        try:
            llm_score = float(response.choices[0].message.content.strip())
        except:
            llm_score = 5.0
        
        combined = 0.25 * init_score + 0.75 * (llm_score / 10)
        scored.append((idx, init_score, llm_score, combined))
    
    scored.sort(key=lambda x: x[3], reverse=True)
    return scored[:final_k]

# Test legal research queries
legal_queries = [
    "data breach liability and damages",
    "employee privacy rights in the workplace",
    "GDPR fines and enforcement"
]

for query in legal_queries:
    print(f"\n{'='*70}")
    print(f"Legal Research Query: '{query}'")
    print(f"{'='*70}\n")
    
    results = legal_search_with_reranking(query)
    
    for rank, (idx, init, llm, combined) in enumerate(results, 1):
        case = legal_docs[idx]
        print(f"{rank}. [{case['id']}] {case['title']}")
        print(f"   Scores: Initial={init:.3f}, LLM={llm:.1f}, Combined={combined:.3f}")
        print(f"   {case['content'][:80]}...\n")

## ❌ Failure Case

When re-ranking fails and performance considerations.

In [ ]:
# Demonstrating re-ranking failures

print("=== FAILURE 1: POOR INITIAL RETRIEVAL ===\n")

# Query that initial retriever fails on
bad_query = "Monty Python comedy sketches"
print(f"Query: '{bad_query}'\n")

# Initial retrieval (will miss the relevant doc due to embedding mismatch)
candidates = initial_retrieval(bad_query, doc_embeddings, documents, top_k=3)
print("Initial candidates:")
for idx, score in candidates:
    print(f"  {score:.4f}: {documents[idx]}")

print("\n⚠️ Issue: The actual 'Monty Python' document wasn't in initial candidates!")
print("   No re-ranking can fix this - garbage in, garbage out.\n")

print("=== FAILURE 2: LATENCY PROBLEMS ===\n")

import time

query = "programming language"
candidates = initial_retrieval(query, doc_embeddings, documents, top_k=10)

print(f"Re-ranking {len(candidates)} candidates with LLM...")
start = time.time()
reranked = llm_rerank(query, candidates, documents)
elapsed = time.time() - start

print(f"Time elapsed: {elapsed:.2f} seconds")
print(f"Average per candidate: {elapsed/len(candidates):.2f} seconds")
print(f"\n⚠️ Issue: LLM-based re-ranking is SLOW (10 candidates = {elapsed:.1f}s)")
print(f"   For 100 candidates: ~{elapsed*10:.0f} seconds!\n")

print("=== FAILURE 3: INCONSISTENT LLM SCORING ===\n")

# Same query-document pair, multiple scoring attempts
test_doc = documents[0]
test_query = "programming language"

print(f"Scoring same pair 3 times:")
scores = []
for i in range(3):
    score = llm_score(test_query, test_doc)
    scores.append(score)
    print(f"  Attempt {i+1}: {score}")

print(f"\nScore variance: {max(scores) - min(scores):.1f}")
print("⚠️ Issue: LLM scoring can be inconsistent even with temperature=0\n")

print("=== SOLUTIONS ===")
print("""
1. Better Initial Retrieval:
   - Use hybrid search (BM25 + semantic)
   - Increase initial candidate pool (top 100-200)
   - Query expansion to improve recall

2. Reduce Latency:
   - Use smaller/faster models for re-ranking
   - Batch scoring requests
   - Cache common query patterns
   - Use dedicated cross-encoder models

3. Improve Consistency:
   - Multiple scoring and average
   - Use dedicated re-ranking models (not general LLM)
   - Fine-tuned models for domain
""")

## 📊 Benchmark Comparison

| Approach | NDCG@10 | Latency | Cost | Best For |
|----------|---------|---------|------|----------|
| **Vector Only** | 0.58 | 50ms | Low | Speed critical |
| **Vector + LLM Rerank (k=10)** | 0.72 | 500ms | Medium | Balanced |
| **Vector + LLM Rerank (k=50)** | 0.78 | 2.5s | High | Quality critical |
| **Vector + Cross-Encoder** | 0.76 | 300ms | Low | Production |
| **Hybrid + Cross-Encoder** | 0.82 | 350ms | Low | Best overall |

### Re-Ranking Model Comparison:
| Model | Parameters | Speed | MRR@10 | Setup |
|-------|------------|-------|--------|-------|
| **GPT-3.5** | 175B | Slow | 0.71 | API only |
| **bge-reranker** | 560M | Fast | 0.68 | Self-hosted |
| **cross-encoder/ms-marco** | 110M | Fast | 0.65 | Self-hosted |
| **Cohere Rerank** | Unknown | Medium | 0.70 | API |

### Key Insights:
- Re-ranking typically improves NDCG by 10-20%
- Cross-encoders offer best speed/quality trade-off
- Initial retrieval quality is crucial (garbage in, garbage out)
- k=50-100 candidates is the sweet spot for most use cases

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              RE-RANKING EXPERIMENT LAB                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Re-Ranking Playground\n")
print("Compare initial retrieval vs. re-ranked results\n")

# Use the Python documents from basic example
print(f"Document collection: {len(documents)} documents about Python\n")

query = input("Enter your search query: ")
initial_k = int(input("Number of initial candidates (default 5): ") or "5")

print(f"\n{'='*70}")
print("STAGE 1: INITIAL VECTOR RETRIEVAL")
print(f"{'='*70}\n")

candidates = initial_retrieval(query, doc_embeddings, documents, top_k=initial_k)

for rank, (idx, score) in enumerate(candidates, 1):
    print(f"{rank}. Score: {score:.4f}")
    print(f"   {documents[idx]}\n")

rerank = input("Re-rank with LLM? (y/n): ").lower() == 'y'

if rerank:
    print(f"\n{'='*70}")
    print("STAGE 2: LLM RE-RANKING")
    print(f"{'='*70}\n")
    print("Scoring candidates with LLM...\n")
    
    reranked = llm_rerank(query, candidates, documents)
    
    print(f"{'Rank':<6}{'Initial':<10}{'LLM':<8}{'Combined':<10} Document{'':<30}")
    print("-" * 70)
    
    for rank, (idx, init, llm, combined) in enumerate(reranked, 1):
        doc_short = documents[idx][:40] + "..." if len(documents[idx]) > 40 else documents[idx]
        print(f"{rank:<6}{init:<10.3f}{llm:<8.1f}{combined:<10.3f} {doc_short}")
    
    # Show rank changes
    print("\n" + "="*70)
    print("RANK CHANGES:")
    print("="*70 + "\n")
    
    initial_ranks = {idx: r for r, (idx, _) in enumerate(candidates, 1)}
    for new_rank, (idx, init, llm, combined) in enumerate(reranked, 1):
        old_rank = initial_ranks[idx]
        change = old_rank - new_rank
        change_str = f"↑{change}" if change > 0 else f"↓{abs(change)}" if change < 0 else "→"
        print(f"{documents[idx][:50]}...")
        print(f"  Initial rank: {old_rank} → Final rank: {new_rank} ({change_str})\n")

## 💡 Tips & Tricks

### Re-Ranking Strategies:

**1. Cascade Re-Ranking:**
```
Stage 1: Vector search → Top 1000
Stage 2: Fast cross-encoder → Top 100
Stage 3: Slow LLM → Top 10
```

**2. Diversity-Aware Re-Ranking (MMR):**
```python
def mmr(query_embedding, doc_embeddings, lambda_param=0.5, top_k=10):
    # Maximal Marginal Relevance
    # Balances relevance vs diversity
    selected = []
    candidates = list(range(len(doc_embeddings)))
    
    while len(selected) < top_k and candidates:
        best_score = -1
        best_idx = None
        
        for idx in candidates:
            relevance = cosine_similarity([query_embedding], [doc_embeddings[idx]])[0][0]
            diversity = max([cosine_similarity([doc_embeddings[idx]], [doc_embeddings[s]])[0][0] 
                           for s in selected] or [0])
            score = lambda_param * relevance - (1 - lambda_param) * diversity
            
            if score > best_score:
                best_score = score
                best_idx = idx
        
        selected.append(best_idx)
        candidates.remove(best_idx)
    
    return selected
```

### Production Best Practices:
- **Cache embeddings** for frequently accessed documents
- **Batch LLM calls** for multiple candidates
- **Use async** for parallel scoring
- **Monitor latency** and adjust k accordingly
- **A/B test** re-ranking impact on user metrics

### When NOT to Use Re-Ranking:
- ⚠️ Ultra-low latency requirements (<100ms)
- ⚠️ Very small document collections (<100 docs)
- ⚠️ Initial retrieval already has high NDCG (>0.85)
- ⚠️ Cost-sensitive applications

## 📚 References

### Research:
- [BERT for Document Ranking (Nogueira & Cho, 2019)](https://arxiv.org/abs/1810.04805)
- [ColBERT: Efficient Retrieval (Khattab & Zaharia, 2020)](https://arxiv.org/abs/2004.12832)
- [Maximal Marginal Relevance (Carbonell & Goldstein, 1998)](https://www.cs.cmu.edu/~jgc/publication/The_Use_of_MMR_Diversity_Based_LTMIR_1998.pdf)

### Documentation:
- [Sentence Transformers Cross-Encoders](https://www.sbert.net/examples/applications/cross-encoder/README.html)
- [Cohere Rerank API](https://docs.cohere.com/docs/rerank-2)
- [Jina AI Reranker](https://jina.ai/reranker)

### Related Techniques:
- Semantic Search (Technique 56)
- Hybrid Retrieval (Technique 57)
- Query Expansion (Technique 59)
- Basic RAG (Technique 53)